In [22]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [23]:
df=pd.read_csv("location_aware_gis_leakage_dataset.csv")
df.tail()

,Pressure,Flow_Rate,Temperature,Vibration,RPM,Operational_Hours,Zone,Block,Pipe,Location_Code,Latitude,Longitude,Leakage_Flag
4995,59.510350,99.516531,95.451740,3.193914,1818.299485,4877,Zone_5,Block_4,Pipe_1,Zone_5_Block_4_Pipe_1,25.194977,55.291271,0
4996,67.114106,50.024825,106.185755,3.688683,1913.754749,2615,Zone_3,Block_2,Pipe_1,Zone_3_Block_2_Pipe_1,25.094080,55.176319,0
4997,91.129102,69.420249,106.561040,3.189177,2290.333658,7004,Zone_4,Block_5,Pipe_2,Zone_4_Block_5_Pipe_2,25.146806,55.237761,0
4998,68.080362,87.436484,95.579454,3.856765,2187.107963,4808,Zone_5,Block_4,Pipe_5,Zone_5_Block_4_Pipe_5,25.195688,55.267045,0
4999,51.519344,89.665827,99.231074,2.190040,2004.675372,1226,Zone_4,Block_5,Pipe_3,Zone_4_Block_5_Pipe_3,25.146501,55.225385,0


In [24]:
df.shape

(5000, 13)

In [25]:
df.columns

Index(['Pressure', 'Flow_Rate', 'Temperature', 'Vibration', 'RPM',
       'Operational_Hours', 'Zone', 'Block', 'Pipe', 'Location_Code',
       'Latitude', 'Longitude', 'Leakage_Flag'],
      dtype='object')

In [26]:
df.columns = df.columns.str.lower()
df.columns

Index(['pressure', 'flow_rate', 'temperature', 'vibration', 'rpm',
       'operational_hours', 'zone', 'block', 'pipe', 'location_code',
       'latitude', 'longitude', 'leakage_flag'],
      dtype='object')

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pressure           5000 non-null   float64
 1   flow_rate          5000 non-null   float64
 2   temperature        5000 non-null   float64
 3   vibration          5000 non-null   float64
 4   rpm                5000 non-null   float64
 5   operational_hours  5000 non-null   int64  
 6   zone               5000 non-null   object 
 7   block              5000 non-null   object 
 8   pipe               5000 non-null   object 
 9   location_code      5000 non-null   object 
 10  latitude           5000 non-null   float64
 11  longitude          5000 non-null   float64
 12  leakage_flag       5000 non-null   int64  
dtypes: float64(7), int64(2), object(4)
memory usage: 507.9+ KB


In [28]:
df.isnull().sum()

pressure             0
flow_rate            0
temperature          0
vibration            0
rpm                  0
operational_hours    0
zone                 0
block                0
pipe                 0
location_code        0
latitude             0
longitude            0
leakage_flag         0
dtype: int64

In [29]:
df.duplicated().sum()


np.int64(0)

In [30]:
df['leakage_flag'].value_counts()

leakage_flag
0    4677
1     323
Name: count, dtype: int64

In [31]:
df.describe()

,pressure,flow_rate,temperature,vibration,rpm,operational_hours,latitude,longitude,leakage_flag
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,60.056019,79.851892,100.052765,3.008258,1994.494314,5488.247800,25.183972,55.248892,0.064600
std,9.964798,15.156556,4.993849,0.501668,295.012393,2611.426293,0.059976,0.047702,0.245843
min,27.587327,21.163996,83.122104,1.071812,903.474045,1000.000000,25.081389,55.146015,0.000000
25%,53.420950,69.701256,96.695568,2.667660,1789.471968,3197.500000,25.145249,55.225614,0.000000
50%,60.134656,79.738242,100.049587,3.009630,1997.594952,5489.000000,25.193184,55.270521,0.000000
75%,66.660106,90.158571,103.377671,3.353307,2195.181932,7748.500000,25.207640,55.285752,0.000000
max,99.262377,132.935828,117.144552,5.239542,3083.405019,9998.000000,25.291945,55.312817,1.000000


In [32]:
df.drop(['zone', 'block', 'pipe', 'location_code'], axis=1, inplace=True)

In [33]:
X = df.drop("leakage_flag", axis=1)
y = df["leakage_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [34]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [35]:
y_train_smote.value_counts()

leakage_flag
0    3742
1    3742
Name: count, dtype: int64

In [36]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

In [37]:
svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train_scaled, y_train_smote)


SVC(random_state=42)

In [38]:
y_pred = svm_model.predict(X_test_scaled)

In [39]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy of SVM: {accuracy:.2f}%".format(accuracy=accuracy * 100))

Accuracy of SVM: 96.50%


In [40]:
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
print("Confusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", class_report)

Confusion Matrix:
 [[902  33]
 [  2  63]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.96      0.98       935
           1       0.66      0.97      0.78        65

    accuracy                           0.96      1000
   macro avg       0.83      0.97      0.88      1000
weighted avg       0.98      0.96      0.97      1000



In [42]:
import joblib

joblib.dump(svm_model, "Water_leakage_model.pkl")

['Water_leakage_model.pkl']